In [0]:
# ============================================================
# nb_bronze_to_silver
# Incremental: reads only today's Bronze parquet
# Merges into Silver Delta table (no duplicates ever)
# Auth: via Cluster Spark Config (no hardcoded keys)
# ============================================================

from pyspark.sql.functions import (
    col, to_date, year, month, quarter, dayofmonth,
    round, when, lit, current_timestamp, count
)
from delta.tables import DeltaTable
from datetime import date

# ── Paths ──────────────────────────────────────────────────
BRONZE_PATH = "abfss://bronze@ecommerceprojectstorage.dfs.core.windows.net/"
SILVER_PATH = "abfss://silver@ecommerceprojectstorage.dfs.core.windows.net/ecommerce_sales/"

# ── Build today's Bronze folder path ───────────────────────
# ADF drops files in bronze/yyyy/MM/dd/ every run
today       = date.today()
bronze_today = (
    f"{BRONZE_PATH}"
    f"{today.year}/"
    f"{today.month:02d}/"
    f"{today.day:02d}/"
)
print(f"Reading Bronze path: {bronze_today}")

# ── Read today's parquet file ───────────────────────────────
try:
    df_raw = spark.read.parquet(bronze_today)
    row_count = df_raw.count()
    print(f"Rows found in Bronze today: {row_count}")
except Exception as e:
    print(f"No file found in Bronze for today: {e}")
    dbutils.notebook.exit("NO_BRONZE_FILE_TODAY")

if row_count == 0:
    print("Bronze file is empty. Exiting.")
    dbutils.notebook.exit("EMPTY_BRONZE_FILE")

# ── CLEANING ───────────────────────────────────────────────

# 1. Drop exact duplicates on order_id
df = df_raw.dropDuplicates(["order_id"])

# 2. Fill nulls with safe defaults
df = df.fillna({
    "city"           : "Unknown",
    "sub_category"   : "Unknown",
    "product_name"   : "Unknown",
    "payment_status" : "Pending",
    "region"         : "Unknown",
    "quantity"       : 0,
    "unit_price"     : 0.0,
    "discount"       : 0.0,
    "sales"          : 0.0,
    "profit"         : 0.0
})

# 3. Parse order_date string DD-MM-YYYY → proper DateType
df = df.withColumn(
    "order_date_parsed",
    to_date(col("order_date"), "dd-MM-yyyy")
)

# 4. Drop rows where date could not be parsed at all
df = df.filter(col("order_date_parsed").isNotNull())

# 5. Drop business-impossible rows
df = df.filter(
    (col("quantity")   >= 0) &
    (col("unit_price") >= 0) &
    (col("sales")      >= 0)
)

# ── TRANSFORMATION ─────────────────────────────────────────
df = df \
    .withColumn("order_year",
        year(col("order_date_parsed"))) \
    .withColumn("order_month",
        month(col("order_date_parsed"))) \
    .withColumn("order_quarter",
        quarter(col("order_date_parsed"))) \
    .withColumn("order_day",
        dayofmonth(col("order_date_parsed"))) \
    .withColumn("net_sales",
        round(col("sales") * (1 - col("discount") / 100), 2)) \
    .withColumn("profit_margin_pct",
        when(col("sales") > 0,
             round(col("profit") / col("sales") * 100, 2)
        ).otherwise(lit(0.0))) \
    .withColumn("revenue_category",
        when(col("sales") >= 100000, lit("High"))
        .when(col("sales") >= 30000,  lit("Medium"))
        .otherwise(lit("Low"))) \
    .withColumn("_silver_load_ts", current_timestamp()) \
    .withColumn("_bronze_source",  lit(bronze_today))

clean_count = df.count()
print(f"Clean rows after transformation: {clean_count}")

# ── WRITE to Silver via Delta MERGE ────────────────────────
# If Silver exists → MERGE (upsert on order_id)
# If Silver missing → write fresh (first run only)

if DeltaTable.isDeltaTable(spark, SILVER_PATH):
    silver_table = DeltaTable.forPath(spark, SILVER_PATH)

    silver_table.alias("tgt").merge(
        df.alias("src"),
        "tgt.order_id = src.order_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

    print(f"Silver MERGE complete — {clean_count} rows upserted")

else:
    # First run — create Silver Delta table with partitioning
    df.write \
        .format("delta") \
        .partitionBy("order_year", "order_month") \
        .mode("overwrite") \
        .save(SILVER_PATH)

    print(f"Silver initial write complete — {clean_count} rows written")

print("\nnb_bronze_to_silver DONE ✅")